# Notebook 03: Prompt Templates for Production

This notebook teaches you to build **reusable, production-ready prompt templates** for classification, summarization, and extraction tasks. You will learn to use LangChain prompt templates and build your own templating system.

## Learning Objectives
- Build reusable prompt templates with variable substitution
- Create templates for classification, summarization, and extraction
- Use LangChain prompt templates
- Design templates that handle edge cases gracefully

In [ ]:
import json
import openai


client = openai.OpenAI()

def generate(prompt, system_prompt=None, temperature=0.3, max_tokens=1024):
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_tokens
    )
    if system_prompt:
        config.system_instruction = system_prompt
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}],
        config=config
    )
    return response.choices[0].message.content

print("Ready.")

## 1. Building Prompt Templates from Scratch

A prompt template is a string with placeholders that get filled at runtime.

In [ ]:
# Simple string-based template
class PromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, **kwargs):
        missing = set(self.input_variables) - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing variables: {missing}")
        return self.template.format(**kwargs)

# Define a classification template
classification_template = PromptTemplate(
    template="""You are a {role} classifier.

Classify the following text into exactly one category.
Categories: {categories}

Text: {text}

Respond with ONLY the category name, nothing else.""",
    input_variables=["role", "categories", "text"]
)

# Use the template
prompt = classification_template.format(
    role="customer support",
    categories="BILLING, TECHNICAL, FEATURE, ACCOUNT",
    text="I can't access my dashboard after the latest update"
)

print("=== GENERATED PROMPT ===")
print(prompt)
print("\n--- Response ---")
print(generate(prompt, temperature=0.0))

## 2. Classification Template

A robust classification template that handles edge cases.

In [ ]:
class ClassificationTemplate:
    def __init__(self, categories, role="analyzer"):
        self.categories = categories
        self.role = role
        self.template = """You are a {role}.

Classify the input text into exactly one of these categories:
{category_list}

Rules:
- Return ONLY the category name
- If the text is ambiguous, choose the MOST LIKELY category
- If the text is empty or unreadable, return: UNCLASSIFIED
- Do not explain your reasoning

Input text:
"{text}"

Category:"""

    def render(self, text):
        category_list = chr(10).join(f"- {c}" for c in self.categories)
        return self.template.format(
            role=self.role,
            category_list=category_list,
            text=text
        )

# Usage
ticket_classifier = ClassificationTemplate(
    categories=["BUG", "FEATURE_REQUEST", "QUESTION", "COMPLAINT", "PRaise"],
    role="customer support ticket classifier"
)

test_tickets = [
    "The app crashes every time I try to upload a file larger than 10MB",
    "Would be nice if you could add a dark mode",
    "How do I export my data to CSV?",
    "This is the worst product I've ever used. Complete waste of money.",
    "Absolutely love the new update! The UI is so much cleaner now.",
    "",  # Edge case: empty input
]

print("=== CLASSIFICATION TEMPLATE ===")
for ticket in test_tickets:
    prompt = ticket_classifier.render(ticket)
    result = generate(prompt, temperature=0.0, max_tokens=20)
    print(f"  {ticket[:50] if ticket else '(empty)':50s} -> {result.strip()}")

## 3. Summarization Template

A template that produces consistent summaries with configurable length and focus.

In [ ]:
class SummarizationTemplate:
    def __init__(self, style="concise"):
        self.style = style
        self.styles = {
            "concise": {
                "instructions": "Summarize in 2-3 sentences. Focus on key facts and numbers.",
                "format": "Return as a single paragraph."
            },
            "bullet": {
                "instructions": "Extract 3-5 key points from the text.",
                "format": "Return as a bullet list, one point per line."
            },
            "executive": {
                "instructions": "Write an executive summary. Lead with the most important finding. Include specific metrics where available.",
                "format": "Return as: TL;DR (one line) followed by 3-5 bullet points."
            },
            "technical": {
                "instructions": "Summarize for a technical audience. Focus on architecture, implementation details, and trade-offs.",
                "format": "Return as a numbered list of technical findings."
            }
        }

    def render(self, text, context=""):
        style_config = self.styles.get(self.style, self.styles["concise"])
        context_block = f"\nContext: {context}\n" if context else ""

        template = f"""Summarize the following text.

{context_block}
Style: {self.style}
Instructions: {style_config['instructions']}
Output format: {style_config['format']}

--- TEXT ---
{text}
--- END TEXT ---"""

        return template

# Test with a sample document
sample_report = """Q3 2024 Financial Performance Report

Revenue grew 23% year-over-year to $12.4M, driven primarily by enterprise deal
closings. The sales team closed 47 new enterprise accounts, up from 31 in Q2.
Average deal size increased to $85K from $72K.

Operating expenses were $9.1M, representing a 15% increase YoY. The increase
was concentrated in R&D (hiring 12 engineers) and marketing (new campaign launch).
Gross margin held steady at 72%.

Cash position remains strong at $28M with 18 months of runway at current burn rate.
The board has approved a $5M credit facility for strategic acquisitions.

Key risks: Enterprise sales cycle lengthening (avg 94 days vs 78 days in Q2),
increasing competition from two well-funded startups, and potential impact of
upcoming regulatory changes on data processing fees."""

print("=== SUMMARIZATION STYLES ===")
for style in ["concise", "bullet", "executive", "technical"]:
    summarizer = SummarizationTemplate(style=style)
    prompt = summarizer.render(sample_report, context="Quarterly board meeting preparation")
    result = generate(prompt, temperature=0.3)
    print(f"\n--- {style.upper()} ---")
    print(result)

## 4. Extraction Template

Templates that extract structured data from unstructured text.

In [ ]:
class ExtractionTemplate:
    def __init__(self, schema, entity_type="entity"):
        self.schema = schema
        self.entity_type = entity_type

    def render(self, text):
        schema_str = json.dumps(self.schema, indent=2)
        return f"""Extract structured {self.entity_type} data from the text below.

Expected JSON schema:
{schema_str}

Rules:
- Return ONLY valid JSON matching the schema above
- Use null for fields that cannot be determined
- Return arrays for repeated fields
- Do not include any text outside the JSON

--- TEXT ---
{text}
--- END TEXT ---

JSON:"""

# Define schemas for different extraction tasks
person_schema = {
    "name": "string",
    "title": "string",
    "company": "string",
    "email": "string or null",
    "phone": "string or null"
}

company_schema = {
    "name": "string",
    "industry": "string",
    "founded": "string or null",
    "headquarters": "string or null",
    "employees": "string (e.g., '100-500') or null",
    "funding_stage": "string or null"
}

# Test extraction
meeting_notes = """Met with Sarah Chen, VP of Engineering at DataFlow Inc.
She's been with the company since 2019 and oversees a team of 45 engineers.
DataFlow is a Series B startup (raised $25M last quarter) based in San Francisco
with about 120 employees. They focus on real-time data pipelines.
Sarah's email is sarah@dataflow.io and her direct line is 415-555-0142."""

# Extract person
person_extractor = ExtractionTemplate(person_schema, entity_type="person")
prompt = person_extractor.render(meeting_notes)
print("=== PERSON EXTRACTION ===")
result = generate(prompt, temperature=0.0)
print(result)

# Extract company
company_extractor = ExtractionTemplate(company_schema, entity_type="company")
prompt = company_extractor.render(meeting_notes)
print("\n=== COMPANY EXTRACTION ===")
result = generate(prompt, temperature=0.0)
print(result)

## 5. LangChain Prompt Templates

LangChain provides a battle-tested template system with additional features like partial variables and composition.

In [ ]:
# Install if needed: pip install langchain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Basic PromptTemplate
template = PromptTemplate(
    input_variables=["category", "text"],
    template="""Classify this {category} text:

"{text}"

Return: positive / negative / neutral"""
)

print("=== LangChain PromptTemplate ===")
print(template.format(category="review", text="Great product, works perfectly!"))

# ChatPromptTemplate with system + human messages
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Respond in {format} format."),
    ("human", "{input}")
])

print("\n=== LangChain ChatPromptTemplate ===")
messages = chat_template.format_messages(
    role="data analyst",
    format="JSON",
    input="Analyze the sales trend: Q1=$1M, Q2=$1.3M, Q3=$1.1M"
)
for msg in messages:
    print(f"[{msg.type}] {msg.content[:80]}...")

In [ ]:
# LangChain with partial variables (for templates that never change)
system_prompt = PromptTemplate(
    input_variables=["format"],
    partial_variables={
        "tone": "professional and concise",
        "constraints": "Do not make up information. Say 'I don't know' if unsure."
    },
    template="""You are a customer support agent.
Tone: {tone}
Constraints: {constraints}
Output format: {format}"""
)

# Now only need to provide the format
print("=== Partial Variables ===")
print(system_prompt.format(format="JSON with 'response' and 'action' fields"))

In [ ]:
# Composition: chain templates together
extraction_prompt = PromptTemplate(
    input_variables=["entity_type", "schema", "text"],
    template="""Extract {entity_type} information from the text.

Schema: {schema}

Text: {text}

Return ONLY valid JSON."""
)

validation_prompt = PromptTemplate(
    input_variables=["extracted_data", "schema"],
    template="""Validate this extracted data against the schema.

Schema: {schema}
Extracted data: {extracted_data}

Return JSON: {{\"valid\": true/false, \"errors\": [\"list of issues\"]}}"""
)

# Simulate a pipeline
text = "Contact John at john@example.com for the API integration."

step1 = extraction_prompt.format(
    entity_type="contact",
    schema=json.dumps({"name": "string", "email": "string", "role": "string"}),
    text=text
)

extracted = generate(step1, temperature=0.0)
print("=== Pipeline Step 1: Extraction ===")
print(extracted)

step2 = validation_prompt.format(
    extracted_data=extracted,
    schema=json.dumps({"name": "string", "email": "string", "role": "string"})
)

validated = generate(step2, temperature=0.0)
print("\n=== Pipeline Step 2: Validation ===")
print(validated)

## 6. Production Template Patterns

### Pattern 1: Template Registry
Centralize all prompts in one place for versioning and management.

In [ ]:
class PromptRegistry:
    """Central registry for all prompt templates."""

    def __init__(self):
        self.templates = {}
        self.versions = {}

    def register(self, name, template, version="1.0"):
        if name not in self.templates:
            self.templates[name] = {}
            self.versions[name] = []
        self.templates[name][version] = template
        self.versions[name].append(version)

    def get(self, name, version=None):
        if name not in self.templates:
            raise KeyError(f"Template '{name}' not found")
        if version is None:
            version = self.versions[name][-1]  # Latest version
        return self.templates[name][version]

    def list_templates(self):
        return {name: self.versions[name][-1] for name in self.templates}

# Usage
registry = PromptRegistry()

registry.register(
    name="classify_ticket",
    template=classification_template,
    version="1.0"
)

registry.register(
    name="classify_ticket",
    template=PromptTemplate(
        template="""V2: Enhanced classifier with confidence scoring.
{text}""",
        input_variables=["text"]
    ),
    version="2.0"
)

print("=== Prompt Registry ===")
print(f"Templates: {registry.list_templates()}")
print(f"Classify ticket latest version: {registry.get('classify_ticket').template[:60]}...")

### Pattern 2: Template with Fallback
Handle failures gracefully by providing fallback behavior.

In [ ]:
class ResilientClassifier:
    def __init__(self, categories, fallback="UNKNOWN"):
        self.categories = categories
        self.fallback = fallback
        self.template = """Classify into exactly one of: {categories}

Input: "{text}"

Return ONLY the category name."""

    def classify(self, text):
        if not text or not text.strip():
            return self.fallback

        prompt = self.template.format(
            categories=", ".join(self.categories),
            text=text[:500]  # Truncate to prevent token overflow
        )

        try:
            result = generate(prompt, temperature=0.0, max_tokens=20)
            result = result.strip().upper()

            # Validate against known categories
            for category in self.categories:
                if category.upper() in result:
                    return category
            return self.fallback

        except Exception:
            return self.fallback

# Test
classifier = ResilientClassifier(
    categories=["BUG", "FEATURE", "QUESTION", "BILLING"],
    fallback="UNCLASSIFIED"
)

test_cases = [
    "App crashes on startup",
    "",  # Empty
    None,  # None
    "A" * 2000,  # Very long input
    "Can I get a refund?",
]

print("=== Resilient Classifier ===")
for text in test_cases:
    result = classifier.classify(text)
    display = text[:40] if text else repr(text)
    print(f"  {display:42s} -> {result}")

## 7. Exercises

### Exercise 1: Build a Multi-Task Template

Create a single template that can handle three different tasks based on a `task_type` variable:
1. **classify** — Classify input into categories
2. **summarize** — Generate a 2-sentence summary
3. **extract** — Extract key entities as JSON

In [ ]:
# YOUR CODE HERE
# Build a multi-task template
# Test it with all three task types

### Exercise 2: Create a Prompt Versioning System

Extend the `PromptRegistry` to:
- Track which version is currently active
- Allow A/B testing between versions
- Log which version produced which output

In [ ]:
# YOUR CODE HERE
# Extend PromptRegistry with A/B testing and logging

### Exercise 3: Build a Pipeline Template

Create a template pipeline that:
1. Takes raw customer feedback
2. Extracts structured data (sentiment, category, urgency)
3. Generates a response based on the extracted data
4. Validates the response format

Each step should be a separate template.

In [ ]:
# YOUR CODE HERE
# Build a 4-step template pipeline
# Test with sample customer feedback

## Key Takeaways

1. **Templates are code** — version them, test them, treat them like any other dependency
2. **Separate concerns** — each template should do one thing well
3. **Handle edge cases** — empty input, invalid format, missing fields
4. **Validate outputs** — never trust raw LLM output in production
5. **Use LangChain** for complex template composition, or build simple ones from scratch
6. **Centralize prompts** — use a registry or config file, not scattered strings